## Logistic regression with 5-fold spatial cross-validation

The model uses elevation, slope, aspect_sin, and aspect_cos. Each time, one spatial fold is used for testing and the other four folds are used for training.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cluster import KMeans

In [47]:
CSV_PATH = Path("/Users/liwei/Desktop/training_matrix_with_absences.csv")
df = pd.read_csv(CSV_PATH)
df.head()

,x_coord,y_coord,presence,elevation,slope,aspect
0,9.370629e+05,-2.602305e+06,1,113.80469,12.086203,229.215260
1,-9.753301e+05,-2.762410e+06,1,524.99460,23.021828,28.830984
2,-9.753301e+05,-2.762410e+06,1,524.99460,23.021828,28.830984
3,-9.753301e+05,-2.762410e+06,1,524.99460,23.021828,28.830984
4,-1.131976e+06,-2.573076e+06,1,537.42487,18.381567,0.392016


Convert the annular slope direction into two available features of the model; otherwise, it is impossible to distinguish between 1 and 359.<br>
When sin is positive, the slope is more eastward. Negative: The slope is more westward.<br>
When cos is positive, the slope is more northerly. Negative: The slope is further south.<br>

In [48]:
aspect_rad = np.deg2rad(df["aspect"])
df["aspect_sin"] = np.sin(aspect_rad)
df["aspect_cos"] = np.cos(aspect_rad)
df.head()

,x_coord,y_coord,presence,elevation,slope,aspect,aspect_sin,aspect_cos
0,9.370629e+05,-2.602305e+06,1,113.80469,12.086203,229.215260,-0.757169,-0.653219
1,-9.753301e+05,-2.762410e+06,1,524.99460,23.021828,28.830984,0.482227,0.876046
2,-9.753301e+05,-2.762410e+06,1,524.99460,23.021828,28.830984,0.482227,0.876046
3,-9.753301e+05,-2.762410e+06,1,524.99460,23.021828,28.830984,0.482227,0.876046
4,-1.131976e+06,-2.573076e+06,1,537.42487,18.381567,0.392016,0.006842,0.999977


1. Use Kmean to group the folds: locations with close coordinate positions are usually more likely to have similar geographical environments and topographic features, such as elevation, slope, and aspect. At the same time, the situation of night parrots may also be similar <br>
2. Another point is that if grouping is done solely based on coordinates, there might be cases where some folds do not have presence or background information, which would prevent the final analysis of the model's performance.<br>
3. Thats's why firstly using presence_sites for classification and find the center points, and then place background_sites into the center points that are close to that center points. <br>
<br>
fit: Find the center point<br>
predict: allocate site into the group of the nearest center point<br>
fit_predict: combine fit and predict<br>

In [50]:
# 1. Separate unique presence and background locations
presence_sites = df[df["presence"] == 1].drop_duplicates(
    subset=["x_coord", "y_coord"]
)[["x_coord", "y_coord", "presence"]].copy()

background_sites = df[df["presence"] == 0].drop_duplicates(
    subset=["x_coord", "y_coord"]
)[["x_coord", "y_coord", "presence"]].copy()

In [51]:
presence_sites.head()

,x_coord,y_coord,presence
0,9.370629e+05,-2.602305e+06,1
1,-9.753301e+05,-2.762410e+06,1
4,-1.131976e+06,-2.573076e+06,1
10,3.140137e+05,-3.550748e+06,1
11,5.129104e+05,-3.165249e+06,1


In [52]:
background_sites.head()

,x_coord,y_coord,presence
55,-5.603582e+05,-2.767932e+06,0
56,-6.818191e+05,-2.897370e+06,0
57,-1.431691e+05,-2.783006e+06,0
58,-1.031542e+06,-2.432518e+06,0
59,-1.254611e+04,-2.439180e+06,0


In [53]:
# 2. Create five spatial folds based on presence locations
kmeans = KMeans(n_clusters=5, n_init=50, random_state=1234)

presence_sites["fold"] = kmeans.fit_predict(
    presence_sites[["x_coord", "y_coord"]]
)

# 3. Assign every background location to its nearest presence-based fold
background_sites["fold"] = kmeans.predict(
    background_sites[["x_coord", "y_coord"]]
)

# 4. Combine the two location tables and check fold
unique_sites = pd.concat(
    [presence_sites, background_sites],
    ignore_index=True
)

fold_check = pd.crosstab(
    unique_sites["fold"],
    unique_sites["presence"]
)

fold_check.columns = ["Background (0)", "Presence (1)"]

print(fold_check)

      Background (0)  Presence (1)
fold                              
0                  5             6
1                 20            13
2                 12             6
3                 12             4
4                  6             6


In [54]:
df = pd.merge(
    df,
    unique_sites[["x_coord", "y_coord", "fold"]],
    on=["x_coord", "y_coord"],
    how="left"
)

print(pd.crosstab(df["fold"], df["presence"]))

presence   0   1
fold            
0          5   8
1         20  30
2         12   7
3         12   4
4          6   6
